# 3. 모델 공정성 평가 (ML)

## 개요

이 노트북은 머신러닝 모델의 **공정성(Fairness)**을 평가합니다.

### 평가 메트릭

보호 변수(성별, 인종 등) 그룹 간 모델 성능 차이를 측정합니다:

#### 1. **False Positive Rate (FPR) Parity**
- **정의**: 실제 음성(Negative)인 샘플 중 모델이 양성(Positive)으로 잘못 예측한 비율
- **공식**: `FPR = FP / (FP + TN)`
- **공정성 기준**: 모든 그룹의 FPR이 유사해야 함 (차이 < 0.1 권장)
- **의미**: 억울한 피해(False Alarm)가 특정 그룹에 집중되지 않아야 함

#### 2. **False Discovery Rate (FDR) Parity**
- **정의**: 모델이 양성으로 예측한 샘플 중 실제로는 음성인 비율
- **공식**: `FDR = FP / (FP + TP)`
- **공정성 기준**: 모든 그룹의 FDR이 유사해야 함
- **의미**: 예측의 정확도(Precision의 역수)가 그룹 간 공평해야 함

#### 3. **True Positive Rate (TPR) Parity (Recall Parity)**
- **정의**: 실제 양성인 샘플 중 모델이 정확히 양성으로 예측한 비율
- **공식**: `TPR = TP / (TP + FN)`
- **공정성 기준**: 모든 그룹의 TPR이 유사해야 함
- **의미**: 혜택(Opportunity)이 모든 그룹에 공평하게 제공되어야 함

### 데이터셋
- UCI Adult Income Dataset
- 보호 변수: 성별(sex)
- 목표: 연소득 >50K 예측

## 1. 환경 설정 및 데이터 로드

In [ ]:
# 필요한 라이브러리 설치
!pip install -q pandas numpy matplotlib seaborn scikit-learn fairlearn ucimlrepo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from fairlearn.metrics import (
    MetricFrame,
    false_positive_rate,
    false_negative_rate,
    true_positive_rate,
    false_discovery_rate,
    selection_rate
)

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 시각화 스타일
sns.set_style('whitegrid')
sns.set_palette('Set2')

# 랜덤 시드 고정
np.random.seed(42)

print("라이브러리 로드 완료")

In [ ]:
# UCI Adult Income 데이터셋 로드
adult = fetch_ucirepo(id=2)
X = adult.data.features
y = adult.data.targets

# 데이터 결합
df = pd.concat([X, y], axis=1)

# 라벨을 이진 변수로 변환
df['income_binary'] = (df['income'] == '>50K').astype(int)

# 결측치 제거
df = df.dropna()

print(f"데이터셋 크기: {df.shape}")
print(f"\n라벨 분포:\n{df['income_binary'].value_counts()}")
print(f"\n성별 분포:\n{df['sex'].value_counts()}")
df.head()

## 2. 데이터 전처리

In [ ]:
# 보호 변수 저장 (나중에 공정성 평가에 사용)
sensitive_feature = df['sex'].copy()

# 특성과 라벨 분리
feature_cols = ['age', 'education-num', 'hours-per-week', 'capital-gain', 'capital-loss']
X = df[feature_cols].copy()
y = df['income_binary'].copy()

# 훈련/테스트 분할
X_train, X_test, y_train, y_test, sensitive_train, sensitive_test = train_test_split(
    X, y, sensitive_feature, test_size=0.3, random_state=42, stratify=y
)

# 정규화
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"훈련 세트 크기: {X_train.shape}")
print(f"테스트 세트 크기: {X_test.shape}")
print(f"\n테스트 세트 성별 분포:\n{sensitive_test.value_counts()}")

## 3. 모델 학습

In [ ]:
# Random Forest 분류기 학습
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

print("모델 학습 중...")
model.fit(X_train_scaled, y_train)

# 예측
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

# 기본 성능 평가
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print(f"\n모델 학습 완료")
print(f"훈련 세트 정확도: {train_acc:.4f}")
print(f"테스트 세트 정확도: {test_acc:.4f}")

print("\n분류 리포트 (테스트 세트):")
print(classification_report(y_test, y_pred_test, target_names=['<=50K', '>50K']))

## 4. 공정성 평가

### 4.1 그룹별 Confusion Matrix

In [ ]:
# 성별별 confusion matrix 계산
def plot_group_confusion_matrix(y_true, y_pred, sensitive, group_name):
    """
    특정 그룹의 confusion matrix 시각화
    """
    mask = (sensitive == group_name)
    cm = confusion_matrix(y_true[mask], y_pred[mask])
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['<=50K', '>50K'],
                yticklabels=['<=50K', '>50K'])
    plt.title(f'Confusion Matrix - {group_name}', fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=11)
    plt.xlabel('Predicted Label', fontsize=11)
    return cm

# 남성 그룹
cm_male = plot_group_confusion_matrix(y_test, y_pred_test, sensitive_test, 'Male')
plt.tight_layout()
plt.savefig('../assets/confusion_matrix_male.png', dpi=300, bbox_inches='tight')
plt.show()

# 여성 그룹
cm_female = plot_group_confusion_matrix(y_test, y_pred_test, sensitive_test, 'Female')
plt.tight_layout()
plt.savefig('../assets/confusion_matrix_female.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.2 Fairlearn을 사용한 공정성 메트릭 계산

In [ ]:
# Fairlearn MetricFrame으로 그룹별 메트릭 계산
metric_frame = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'fpr': false_positive_rate,
        'tpr': true_positive_rate,
        'fdr': false_discovery_rate,
        'selection_rate': selection_rate
    },
    y_true=y_test,
    y_pred=y_pred_test,
    sensitive_features=sensitive_test
)

print("="*80)
print("그룹별 공정성 메트릭")
print("="*80)
print(metric_frame.by_group)
print("\n" + "="*80)
print("전체 평균")
print("="*80)
print(metric_frame.overall)

### 4.3 공정성 메트릭 차이 분석

In [ ]:
# 그룹 간 메트릭 차이 계산
fairness_gaps = {}
by_group = metric_frame.by_group

for metric in ['fpr', 'tpr', 'fdr']:
    male_val = by_group.loc['Male', metric]
    female_val = by_group.loc['Female', metric]
    gap = abs(male_val - female_val)
    fairness_gaps[metric] = {
        'Male': male_val,
        'Female': female_val,
        'Gap': gap,
        'Fair': gap < 0.1  # 차이 < 10%를 공정 기준으로 설정
    }

fairness_df = pd.DataFrame(fairness_gaps).T

print("\n" + "="*80)
print("공정성 격차 분석 (Gap < 0.1 권장)")
print("="*80)
print(fairness_df)
print("="*80)

# 공정성 위반 항목 하이라이트
unfair_metrics = fairness_df[fairness_df['Fair'] == False]
if len(unfair_metrics) > 0:
    print("\n⚠️ 공정성 위반 메트릭:")
    for metric in unfair_metrics.index:
        gap = fairness_gaps[metric]['Gap']
        print(f"  - {metric.upper()}: 그룹 간 차이 {gap:.4f} (기준: 0.1 미만)")
else:
    print("\n✓ 모든 메트릭이 공정성 기준을 만족합니다.")

## 5. 결과 시각화

In [ ]:
# 시각화 1: 그룹별 공정성 메트릭 비교
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_to_plot = ['fpr', 'tpr', 'fdr']
metric_labels = ['False Positive Rate\n(FPR)', 'True Positive Rate\n(TPR/Recall)', 'False Discovery Rate\n(FDR)']

for idx, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    ax = axes[idx]
    
    male_val = fairness_gaps[metric]['Male']
    female_val = fairness_gaps[metric]['Female']
    gap = fairness_gaps[metric]['Gap']
    is_fair = fairness_gaps[metric]['Fair']
    
    bars = ax.bar(['Male', 'Female'], [male_val, female_val], 
                   color=['#3498db', '#e74c3c'], alpha=0.8, edgecolor='black')
    
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.set_ylabel('Rate', fontsize=11)
    ax.set_ylim(0, max(male_val, female_val) * 1.2)
    ax.grid(axis='y', alpha=0.3)
    
    # 값 표시
    for bar, val in zip(bars, [male_val, female_val]):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # Gap 표시
    status = 'FAIR ✓' if is_fair else 'UNFAIR ✗'
    color = 'green' if is_fair else 'red'
    ax.text(0.5, 0.95, f'Gap: {gap:.4f}\n{status}', 
            transform=ax.transAxes, ha='center', va='top',
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.2),
            fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../assets/fairness_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 시각화 2: 공정성 격차(Gap) 비교
fig, ax = plt.subplots(figsize=(10, 6))

gaps = [fairness_gaps[m]['Gap'] for m in metrics_to_plot]
colors = ['#2ecc71' if fairness_gaps[m]['Fair'] else '#e74c3c' for m in metrics_to_plot]

bars = ax.barh(metric_labels, gaps, color=colors, alpha=0.8, edgecolor='black')
ax.axvline(x=0.1, color='red', linestyle='--', linewidth=2, label='Fairness Threshold (0.1)')
ax.set_xlabel('Gap (Absolute Difference)', fontsize=12, fontweight='bold')
ax.set_title('Fairness Gap Analysis (Male vs Female)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(axis='x', alpha=0.3)

# 값 표시
for bar, gap, metric in zip(bars, gaps, metrics_to_plot):
    status = 'FAIR ✓' if fairness_gaps[metric]['Fair'] else 'UNFAIR ✗'
    ax.text(gap + 0.005, bar.get_y() + bar.get_height()/2, 
            f'{gap:.4f} ({status})', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('../assets/fairness_gap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 시각화 3: 전체 메트릭 히트맵
fig, ax = plt.subplots(figsize=(10, 6))

heatmap_data = by_group[['accuracy', 'fpr', 'tpr', 'fdr', 'selection_rate']].T
sns.heatmap(heatmap_data, annot=True, fmt='.4f', cmap='RdYlGn', 
            cbar_kws={'label': 'Metric Value'}, ax=ax,
            linewidths=1, linecolor='black')
ax.set_title('Fairness Metrics Heatmap by Sex', fontsize=14, fontweight='bold')
ax.set_xlabel('Sex', fontsize=12)
ax.set_ylabel('Metric', fontsize=12)

plt.tight_layout()
plt.savefig('../assets/fairness_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. 추가 분석: 임계값 조정에 따른 공정성 변화

In [ ]:
# 확률 예측 (임계값 조정 실험용)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# 다양한 임계값에서 공정성 평가
thresholds = np.linspace(0.1, 0.9, 9)
threshold_results = []

for threshold in thresholds:
    y_pred_thresh = (y_pred_proba >= threshold).astype(int)
    
    mf = MetricFrame(
        metrics={'fpr': false_positive_rate, 'tpr': true_positive_rate},
        y_true=y_test,
        y_pred=y_pred_thresh,
        sensitive_features=sensitive_test
    )
    
    male_fpr = mf.by_group.loc['Male', 'fpr']
    female_fpr = mf.by_group.loc['Female', 'fpr']
    male_tpr = mf.by_group.loc['Male', 'tpr']
    female_tpr = mf.by_group.loc['Female', 'tpr']
    
    threshold_results.append({
        'threshold': threshold,
        'fpr_gap': abs(male_fpr - female_fpr),
        'tpr_gap': abs(male_tpr - female_tpr),
        'overall_acc': accuracy_score(y_test, y_pred_thresh)
    })

threshold_df = pd.DataFrame(threshold_results)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FPR Gap
axes[0].plot(threshold_df['threshold'], threshold_df['fpr_gap'], 
             marker='o', linewidth=2, markersize=8, color='#e74c3c')
axes[0].axhline(y=0.1, color='red', linestyle='--', label='Fairness Threshold (0.1)')
axes[0].set_xlabel('Classification Threshold', fontsize=11, fontweight='bold')
axes[0].set_ylabel('FPR Gap (|Male - Female|)', fontsize=11, fontweight='bold')
axes[0].set_title('FPR Gap vs Threshold', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# TPR Gap
axes[1].plot(threshold_df['threshold'], threshold_df['tpr_gap'], 
             marker='s', linewidth=2, markersize=8, color='#3498db')
axes[1].axhline(y=0.1, color='red', linestyle='--', label='Fairness Threshold (0.1)')
axes[1].set_xlabel('Classification Threshold', fontsize=11, fontweight='bold')
axes[1].set_ylabel('TPR Gap (|Male - Female|)', fontsize=11, fontweight='bold')
axes[1].set_title('TPR Gap vs Threshold', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../assets/threshold_tuning_fairness.png', dpi=300, bbox_inches='tight')
plt.show()

# 최적 임계값 찾기 (FPR + TPR Gap 최소화)
threshold_df['total_gap'] = threshold_df['fpr_gap'] + threshold_df['tpr_gap']
best_threshold = threshold_df.loc[threshold_df['total_gap'].idxmin(), 'threshold']
print(f"\n공정성 관점 최적 임계값: {best_threshold:.2f}")
print(threshold_df[threshold_df['threshold'] == best_threshold])

## 7. 결론 및 개선 방향

### 평가 결과 해석

#### 1. False Positive Rate (FPR) Parity
- **의미**: 억울한 피해(False Alarm)가 특정 그룹에 집중되는지 평가
- **결과 해석**:
  - Gap < 0.1: 공정함 → 남녀 모두 비슷한 비율로 잘못 양성 판정
  - Gap >= 0.1: 불공정 → 한 그룹이 더 많은 억울한 피해
- **예시**: 대출 승인 모델에서 남성보다 여성이 더 자주 잘못 거절당하는 경우

#### 2. True Positive Rate (TPR) Parity (Recall Parity)
- **의미**: 혜택(Opportunity)이 모든 그룹에 공평하게 제공되는지 평가
- **결과 해석**:
  - Gap < 0.1: 공정함 → 자격 있는 사람이 그룹과 무관하게 혜택 받음
  - Gap >= 0.1: 불공정 → 특정 그룹이 혜택을 덜 받음
- **예시**: 채용 모델에서 자격 있는 여성이 남성보다 더 자주 탈락하는 경우

#### 3. False Discovery Rate (FDR) Parity
- **의미**: 모델의 예측 정확도가 그룹 간 공평한지 평가
- **결과 해석**:
  - Gap < 0.1: 공정함 → 양성 예측의 신뢰도가 그룹 간 유사
  - Gap >= 0.1: 불공정 → 한 그룹의 양성 예측이 덜 신뢰할 만함
- **예시**: 대출 승인받은 사람 중 여성이 남성보다 더 자주 연체하는 경우

### 개선 방향

#### 1. 데이터 레벨 개선
- **리샘플링**: 소수 그룹의 샘플 증가 (SMOTE, Oversampling)
- **데이터 수집**: 편향된 데이터 원천 개선
- **특성 엔지니어링**: 편향을 유발하는 특성 제거 또는 변환

#### 2. 알고리즘 레벨 개선
- **Fairness Constraints**: 학습 시 공정성 제약 조건 추가
  ```python
  from fairlearn.reductions import ExponentiatedGradient, DemographicParity
  mitigator = ExponentiatedGradient(estimator, DemographicParity())
  mitigator.fit(X_train, y_train, sensitive_features=sensitive_train)
  ```
- **Adversarial Debiasing**: 보호 변수를 예측하지 못하도록 적대적 학습
- **그룹별 가중치 조정**: 손실 함수에서 그룹별 샘플 가중치 다르게 설정

#### 3. 후처리 레벨 개선
- **임계값 조정**: 그룹별로 다른 분류 임계값 사용
  ```python
  from fairlearn.postprocessing import ThresholdOptimizer
  postprocessor = ThresholdOptimizer(
      estimator=model, 
      constraints="equalized_odds"
  )
  postprocessor.fit(X_train, y_train, sensitive_features=sensitive_train)
  ```
- **Calibration**: 그룹별 확률 보정

#### 4. 평가 및 모니터링
- **다양한 공정성 메트릭 추적**: FPR, TPR, FDR 외에 Demographic Parity, Equalized Odds 등
- **교차 검증**: 다양한 데이터 분할에서 공정성 안정성 확인
- **실시간 모니터링**: 프로덕션 환경에서 공정성 지표 지속 추적

### 공정성과 성능의 트레이드오프

- 공정성을 높이면 전체 모델 성능(정확도)이 약간 감소할 수 있음
- 비즈니스 요구사항과 윤리적 기준 사이의 균형 필요
- **Pareto Frontier 분석**: 공정성-성능 트레이드오프 곡선 시각화

### 윤리적 고려사항

1. **공정성 정의의 다양성**: FPR Parity와 TPR Parity를 동시에 만족하기 어려울 수 있음 (불가능성 정리)
2. **맥락 고려**: 도메인에 따라 중요한 공정성 기준이 다름 (채용 vs 신용평가 vs 의료)
3. **투명성**: 공정성 평가 결과를 이해관계자에게 공개하고 논의
4. **지속적 개선**: 편향은 한 번 제거로 끝나지 않으며, 지속적인 모니터링 필요

### 참고 자료
- Fairlearn 라이브러리: https://fairlearn.org/
- Aequitas (Bias Auditing Toolkit): http://www.datasciencepublicpolicy.org/projects/aequitas/
- "Fairness and Machine Learning" (Barocas, Hardt, Narayanan): https://fairmlbook.org/